# 02. Recuperación y contraste de explicitud

Las cuatro estrategias se comparan sobre los mismos rankings y qrels de la campaña final. Se recalculan las métricas con los scripts de la entrega y se contrastan los CSV con las referencias publicadas.

Run All crea un destino nuevo para este análisis. No ejecuta los modelos de recuperación ni modifica sus rankings.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

In [2]:
IR = ROOT / 'outputs/local_runs/explicitness_v3'
REFERENCE = ROOT / 'outputs/local_runs/academic_corrections_20260930/retrieval'
RUN = new_run('recuperacion')
print('Salidas:', RUN.relative_to(ROOT))

Salidas: outputs/local_runs/final_experiment_20260930/REPLAY_notebook_recuperacion_45e4bfa9e2bc


## Candidatos y orden

BM25, dense, RRF y reranking tienen diez candidatos por formulación. En esta campaña, el reranker cambia el orden de los candidatos de RRF, no el conjunto de diez fragmentos. Por eso Recall@10 y Precision@10 deben coincidir entre ambos, aunque las métricas de orden puedan cambiar.

Algunos registros de ranking conservan el estado de anotación que existía al crearlos. La referencia vigente es qrels_final_v3.jsonl, no aquel campo histórico.

In [3]:
runs = {method: pd.DataFrame(read_jsonl(IR / 'runs' / f'explicitness_v3_{method}.jsonl'))
        for method in ['bm25', 'dense', 'rrf', 'rerank']}
for method, frame in runs.items():
    assert len(frame) == 520 and frame['qid'].nunique() == 52
    assert frame.groupby('qid').size().eq(10).all()
    assert not frame.duplicated(['qid', 'chunk_id']).any()
rrf_sets = runs['rrf'].groupby('qid')['chunk_id'].agg(set)
rerank_sets = runs['rerank'].groupby('qid')['chunk_id'].agg(set)
assert all(rrf_sets[qid] == rerank_sets[qid] for qid in rrf_sets.index)
display(pd.DataFrame([{'estrategia': method, 'formulaciones': frame['qid'].nunique(), 'filas_ranking': len(frame)}
                      for method, frame in runs.items()]))

,estrategia,formulaciones,filas_ranking
0,bm25,52,520
1,dense,52,520
2,rrf,52,520
3,rerank,52,520


## Recálculo de las métricas

El evaluador utiliza relevancia estricta para las métricas binarias e ideal global para nDCG. La tabla que se muestra después procede del directorio recién generado.

In [4]:
run_script('scripts/evaluate_retrieval_v3.py', '--run-dir', IR,
           '--qrels', IR / 'qrels_final_v3.jsonl', '--methods', 'bm25,dense,rrf,rerank',
           '--output-dir', RUN / 'retrieval', log_dir=RUN)
checked_ir = compare_csv(RUN / 'retrieval', REFERENCE / 'evaluation')
metrics = pd.read_csv(RUN / 'retrieval/aggregate_metrics.csv')
display(metrics)

,cohort,method,k,n_base,recall,precision,mrr,ndcg
0,all_pairs,bm25,5,26,0.372817,0.303846,0.544872,0.682949
1,all_pairs,bm25,10,26,0.506525,0.225000,0.547009,0.706239
2,all_pairs,dense,5,26,0.382601,0.315385,0.520833,0.645241
3,all_pairs,dense,10,26,0.517521,0.226923,0.525717,0.677239
4,all_pairs,rerank,5,26,0.429640,0.350000,0.596154,0.733293
5,all_pairs,rerank,10,26,0.547550,0.246154,0.599359,0.734913
6,all_pairs,rrf,5,26,0.426557,0.338462,0.572756,0.718886
7,all_pairs,rrf,10,26,0.547550,0.246154,0.572756,0.728251
8,grade2_eligible,bm25,5,20,0.484663,0.395000,0.708333,0.695911
9,grade2_eligible,bm25,10,20,0.658482,0.292500,0.711111,0.732881


## Diferencias explícita–implícita

El contraste de recuperación calcula **explícita menos implícita** dentro de cada pregunta base. Un valor positivo indica mejor resultado con la formulación explícita. El test de signos utiliza la dirección de las diferencias y descarta los empates; se muestran sus recuentos y el ajuste Holm junto al valor p.

In [5]:
run_script('scripts/paired_explicitness_v3.py', '--metrics-dir', RUN / 'retrieval',
           '--historical-metrics-dir', IR / 'retrieval_metrics_v3',
           '--output-dir', RUN / 'paired', log_dir=RUN)
checked_paired = compare_csv(RUN / 'paired', REFERENCE / 'paired')
paired = pd.read_csv(RUN / 'paired/paired_explicitness_v3.csv')
display(paired.loc[paired['k'] == 10,
    ['method', 'metric', 'mean_diff', 'pos', 'neg', 'ties', 'n_effective', 'p_sign', 'p_holm']])

,method,metric,mean_diff,pos,neg,ties,n_effective,p_sign,p_holm
16,bm25,recall,0.135974,11,2,13,13,0.022461,0.336914
17,bm25,precision,0.057692,11,2,13,13,0.022461,0.359375
18,bm25,mrr,0.002137,6,4,16,10,0.753906,1.000000
19,bm25,ndcg,0.096471,19,6,1,25,0.014633,0.234133
20,dense,recall,0.117796,9,4,13,13,0.266846,1.000000
21,dense,precision,0.053846,9,4,13,13,0.266846,1.000000
22,dense,mrr,0.113400,5,4,17,9,1.000000,1.000000
23,dense,ndcg,0.100172,18,8,0,26,0.075519,0.688477
24,rerank,recall,0.141071,11,3,12,14,0.057373,0.688477
25,rerank,precision,0.061538,11,3,12,14,0.057373,0.860596


La sensibilidad restringe el análisis a las bases con evidencia de grado 2. Los archivos de diferencias por pregunta permiten revisar si la media depende de pocos casos; las diferencias entre efectos de métodos son descriptivas.

In [6]:
display(pd.read_csv(RUN / 'paired/paired_grade2_eligible.csv'))
display(pd.read_csv(RUN / 'paired/between_method_effect_differences.csv'))

,method,k,metric,mean_diff,pos,neg,ties,p_sign,n_pairs,n_effective,family,family_size,p_holm
0,bm25,5,recall,0.103175,9,4,7,0.266846,20,13,primary,16,1.000000
1,bm25,5,precision,0.070000,9,4,7,0.266846,20,13,complementary,16,1.000000
2,bm25,5,mrr,0.008333,6,4,10,0.753906,20,10,complementary,16,1.000000
3,bm25,5,ndcg,0.061065,12,5,3,0.143463,20,17,primary,16,1.000000
4,dense,5,recall,0.139563,9,2,9,0.065430,20,11,primary,16,0.860596
5,dense,5,precision,0.100000,9,2,9,0.065430,20,11,complementary,16,0.860596
6,dense,5,mrr,0.145833,5,3,12,0.726562,20,8,complementary,16,1.000000
7,dense,5,ndcg,0.106365,12,6,2,0.237885,20,18,primary,16,1.000000
8,rerank,5,recall,0.109881,9,3,8,0.145996,20,12,primary,16,1.000000
9,rerank,5,precision,0.090000,9,3,8,0.145996,20,12,complementary,16,1.000000


,cohort,method_a,method_b,k,metric,mean_effect_a_minus_b,n_pairs,pos,neg,ties,n_effective
0,all_pairs,bm25,dense,5,recall,-2.799145e-02,26,3,8,15,11
1,all_pairs,bm25,dense,5,precision,-2.307692e-02,26,4,8,14,12
2,all_pairs,bm25,dense,5,mrr,-1.057692e-01,26,4,8,14,12
3,all_pairs,bm25,dense,5,ndcg,-5.718704e-02,26,7,17,2,24
4,all_pairs,bm25,dense,10,recall,1.817766e-02,26,6,5,15,11
5,all_pairs,bm25,dense,10,precision,3.846154e-03,26,7,5,14,12
6,all_pairs,bm25,dense,10,mrr,-1.112637e-01,26,4,8,14,12
7,all_pairs,bm25,dense,10,ndcg,-3.700396e-03,26,11,14,1,25
8,all_pairs,bm25,rerank,5,recall,-5.158730e-03,26,5,6,15,11
9,all_pairs,bm25,rerank,5,precision,-1.538462e-02,26,5,6,15,11


## Comparación con las referencias

La comprobación exige el mismo conjunto de CSV y el mismo contenido binario. No compara los tiempos de ejecución ni vuelve a ejecutar inferencia.

In [7]:
checked = pd.DataFrame([dict(etapa='recuperacion', **row) for row in checked_ir] +
                       [dict(etapa='explicitud', **row) for row in checked_paired])
assert checked['identico'].all()
display(checked)

,etapa,archivo,identico
0,recuperacion,aggregate_metrics.csv,True
1,recuperacion,before_after.csv,True
2,recuperacion,per_base.csv,True
3,recuperacion,per_query.csv,True
4,explicitud,before_after_paired.csv,True
5,explicitud,between_method_effect_differences.csv,True
6,explicitud,paired_differences_by_base.csv,True
7,explicitud,paired_explicitness_v3.csv,True
8,explicitud,paired_grade2_eligible.csv,True


Una mejora de ordenación no implica mayor cobertura. Estas métricas describen la recuperación de evidencia; la calidad de las respuestas generadas se examina por separado.